# **CellTracksColab - TrackMate - Plate**
---

Colab Notebook for Analyzing Migration Tracks generated by [TrackMate](https://imagej.net/plugins/trackmate/). This notebook is particularly suited for handling and interpreting data structured in a plate format, commonly produced by incubator microscopes like Incucytes.


# **Part 0. Before getting started**
---

<font size = 5>**Important notes**</font>

---
<font size = 5>**Data Requirements for Analysis**</font>

Be advised of one significant limitation inherent to this notebook.

<font size = 4 color="red">**Part2 does not support Track splitting**</font>. For users aiming to compute additional track metrics within this environment, it is crucial to disable track splitting in TrackMate.

It’s important to clarify that the absence of track splitting support does not hinder the notebook's ability to compile and display results in part 3 of the analysis process.

---

<font size = 5>**Data Organization**</font>

**Overview**

To ensure smooth processing and analysis of your tracking data using TrackMate outputs, it is crucial to organize your files in a specific structure. This organization allows for automated identification and loading of relevant files, enabling efficient data handling.

**File Naming Conventions**

Your files should follow a consistent naming scheme that includes the well number and the field of view (FOV) number. This approach facilitates the differentiation of various experimental conditions and replicates. Ensure that:

- Track files end with `-tracks.csv`.
- Spot tables end with `-spots`.

**File Structure**

Organize your files in a single folder, ideally named to reflect the experiment or dataset. Here's an example of a well-structured data directory:

- 📁 **Experiments** `[Folder_path]`
  - 📄 `B02_01_tracking-spots.csv`
  - 📄 `B02_01_tracking-tracks.csv`
  - 📄 `B02_02_tracking-spots.csv`
  - 📄 `B02_02_tracking-tracks.csv`
  - 📄 `B03_01_tracking-spots.csv`
  - 📄 `B03_01_tracking-tracks.csv`
  - 📄 `B03_02_tracking-spots.csv`
  - 📄 `B03_02_tracking-tracks.csv`  

In this structure:
- `B02`, `B03`, etc., represent the well numbers.
- `01`, `02`, etc., indicate the field of view numbers.

**Data Interpretation**

By default, the system interprets:
- **Well Number**: Represents different experimental conditions.
- **Field of View Number**: Acts as a repeat for each condition.

**Custom Mapping**

You have the flexibility to map well numbers to specific experimental conditions. This mapping can be done in a designated cell within the notebook.


In [ ]:
# @markdown #MIT License

print("""
**MIT License**

Copyright (c) 2023 Guillaume Jacquemet

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.""")

--------------------------------------------------------
# **Part 1. Prepare the session and load the data**
--------------------------------------------------------


## **1.1 Load key dependencies**
---


In [ ]:
#@markdown ##Play to load the dependancies

import sys

# Current version of the notebook the user is running
current_version = "1.1.0"
notebook_name = 'CellTracksColab_TrackMate_Plate'

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

user = 'CellMigrationLab'  # Replace with your GitHub username
repo = 'CellTracksColab'

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    # 1. Clone the repository from GitHub

    # We use the PEP 508 syntax: "package[extras] @ url"
    # !pip install -q "nucleisky[all] @ git+https://{token}@github.com/{user}/{repo}.git"
    !git clone https://github.com/{user}/{repo}.git

    sys.path.append("../")
    sys.path.append(f"{repo}/")

    # 2. Install other dependencies
    %pip -q install pandas scikit-learn
    %pip -q install plotly
    %pip -q install tqdm

    # 3. Mount Google Drive
    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")
else:
    sys.path.append("../")

    # Fallback for local environments
    print("✅ Local environment detected. Assuming dependencies are already installed.")

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import requests
import base64
import yaml
import os

from ipywidgets import Dropdown, interact,Layout, VBox, Button
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, clear_output
from scipy.spatial.distance import pdist
from tqdm.notebook import tqdm

import celltracks
from celltracks import *
from celltracks.Track_Plots import *
from celltracks.BoxPlots_Statistics import *
from celltracks.Track_Metrics import *
from celltracks.Distance_to_ROI import *

try:
    output_area = widgets.HTML()
        
    # Online version checking file path
    version_file_path = "notebooks/notebook_latest_versions.yaml"
    version_url = f"https://api.github.com/repos/{user}/{repo}/contents/{version_file_path}"

    headers = {"Accept": "application/vnd.github.v3+json"}
    version_response = requests.get(version_url, headers=headers)

    # Check the response status
    if version_response.status_code == 200:
        content = version_response.json()['content']
        decoded_content = base64.b64decode(content).decode('utf-8')
        config = yaml.safe_load(decoded_content)
        latest_version = config.get(notebook_name, "")

        output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                            f"<b>Latest version available:</b> `{latest_version}`<br>")

        if latest_version == "":
            output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
        elif current_version == latest_version:
            output_area.value += "✅ This notebook is up-to-date.<br>"
        else:
            output_area.value += f"⚠️ A new version of this notebook is available."
    else:
        output_area.value += "⚠️ Could not retrieve the version file.<br>"

    display(output_area)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))

print("✅ Environment Ready")

## **1.2. Compile your data or load existing dataframes**
---

 Please ensure that your data is properly organised (see above)


In [ ]:
#@markdown ##Run this cell to display the UI for loading your dataset

# -----------------------------
# Widget UI
# -----------------------------

Folder_path_widget = widgets.Text(
    value="",
    placeholder="Path to dataset folder",
    description="Folder path:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "120px"}
)

Results_Folder_widget = widgets.Text(
    value="./Results",  # Local equivalent of "/content/Results"
    placeholder="Path to results folder",
    description="Results folder:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Load tracking data",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Helper functions
# -----------------------------

def populate_columns(df, filepath):
    # Extract the parts of the file path
    filename = os.path.basename(filepath)
    well_info = filename[0:3]
    fov_info = filename.split('_')[1]
    filename_without_extension = os.path.splitext(os.path.basename(filepath))[0]
    df['File_name'] = remove_suffix(filename_without_extension)
    # Add well and FOV information as new columns to the DataFrame
    df['Well'] = well_info
    df['FOV'] = fov_info

    df['Condition'] = well_info  # Populate 'Condition' with the parent folder name
    df['experiment_nb'] = fov_info  # Populate 'Repeat' with the folder name

    return df

def load_and_populate(file_pattern, usecols=None, chunksize=100000):
    df_list = []
    pattern = re.compile(file_pattern)  # Compile the file pattern to a regex object
    files_to_process = []

    # First, list all the files we'll be processing
    for dirpath, dirnames, filenames in os.walk(Folder_path):
        for filename in filenames:
            if pattern.match(filename):  # Check if the filename matches the file pattern
                filepath = os.path.join(dirpath, filename)
                files_to_process.append(filepath)

    # Metadata list
    metadata_list = []

    # Create a tqdm instance for progress tracking
    for filepath in tqdm(files_to_process, desc="Processing Files"):
        # Get the expected number of rows in the file (subtracting header rows)
        expected_rows = sum(1 for row in open(filepath)) - 4

        # Get file size
        file_size = os.path.getsize(filepath)

        # Add to the metadata list
        metadata_list.append({
            'filename': os.path.basename(filepath),
            'expected_rows': expected_rows,
            'file_size': file_size
        })

        chunked_reader = pd.read_csv(filepath, skiprows=[1, 2, 3], usecols=usecols, chunksize=chunksize)

        for chunk in chunked_reader:
            processed_chunk = populate_columns(chunk, filepath)
            df_list.append(processed_chunk)

    if not df_list:  # if df_list is empty, return an empty DataFrame
        print(f"No files found with pattern: {file_pattern}")
        return pd.DataFrame(), metadata_list

    merged_df = pd.concat(df_list, ignore_index=True)
    # Verify the total rows in the merged dataframe matches the total expected rows from metadata
    total_expected_rows = sum(item['expected_rows'] for item in metadata_list)
    if len(merged_df) != total_expected_rows:
      print(f"Warning: Mismatch in total rows. Expected {total_expected_rows}, found {len(merged_df)} in the merged dataframe.")
    else:
      print(f"Success: The processed dataframe matches the metadata. Total rows: {len(merged_df)}")
    return merged_df, metadata_list


def sort_and_generate_repeat(merged_df):
    merged_df.sort_values(['Condition', 'experiment_nb'], inplace=True)
    merged_df = merged_df.groupby('Condition', group_keys=False).apply(generate_repeat)
    return merged_df

def generate_repeat(group):
    unique_experiment_nbs = sorted(group['experiment_nb'].unique())
    experiment_nb_to_repeat = {experiment_nb: i+1 for i, experiment_nb in enumerate(unique_experiment_nbs)}
    group['Repeat'] = group['experiment_nb'].map(experiment_nb_to_repeat)
    return group

def remove_suffix(filename):
    suffixes_to_remove = ["-tracks", "-spots"]
    for suffix in suffixes_to_remove:
        if filename.endswith(suffix):
            filename = filename[:-len(suffix)]
            break
    return filename


def validate_tracks_df(df):
    """Validate the tracks dataframe for necessary columns and data types."""
    required_columns = ['TRACK_ID']
    for col in required_columns:
        if col not in df.columns:
            print(f"Error: Column '{col}' missing in tracks dataframe.")
            return False

    # Additional data type checks or value ranges can be added here
    return True

def validate_spots_df(df):
    """Validate the spots dataframe for necessary columns and data types."""
    required_columns = ['TRACK_ID', 'POSITION_X', 'POSITION_Y', 'POSITION_Z', 'POSITION_T']
    for col in required_columns:
        if col not in df.columns:
            print(f"Error: Column '{col}' missing in spots dataframe.")
            return False

    # Additional data type checks or value ranges can be added here
    return True

def check_unique_id_match(df1, df2):
    df1_ids = set(df1['Unique_ID'])
    df2_ids = set(df2['Unique_ID'])

    # Check if the IDs in the two dataframes match
    if df1_ids == df2_ids:
        print("The Unique_ID values in both dataframes match perfectly!")
    else:
        missing_in_df1 = df2_ids - df1_ids
        missing_in_df2 = df1_ids - df2_ids

        if missing_in_df1:
            print(f"There are {len(missing_in_df1)} Unique_ID values present in the second dataframe but missing in the first.")
            print("Examples of these IDs are:", list(missing_in_df1)[:5])

        if missing_in_df2:
            print(f"There are {len(missing_in_df2)} Unique_ID values present in the first dataframe but missing in the second.")
            print("Examples of these IDs are:", list(missing_in_df2)[:5])

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global Folder_path
    global Results_Folder
    global merged_tracks_df
    global merged_spots_df
    global tracks_metadata

    output.value = "🔁 Loading dataset ..."
    process_output.clear_output(wait=True)

    # Read widget values
    Folder_path = Folder_path_widget.value
    Results_Folder = Results_Folder_widget.value

    if not Results_Folder:
        Results_Folder = "./Results"

    # Check if the results folder exists, if not create it
    if not os.path.exists(Results_Folder):
        os.makedirs(Results_Folder, exist_ok=True)

    with process_output:
        print(f"Result folder is located at: {Results_Folder}")

        if Folder_path:
            merged_tracks_df, tracks_metadata = load_and_populate(r'.*tracks.*\.csv')

            if not validate_tracks_df(merged_tracks_df):
                print("Error: Validation failed for merged tracks dataframe.")
            else:
                merged_tracks_df = sort_and_generate_repeat(merged_tracks_df)
                merged_tracks_df['Unique_ID'] = merged_tracks_df['File_name'] + "_" + merged_tracks_df['TRACK_ID'].astype(str)
                save_dataframe_with_progress(
                    merged_tracks_df,
                    os.path.join(Results_Folder, 'merged_Tracks.csv'),
                    desc="Saving Tracks"
                )

            merged_spots_df, tracks_metadata = load_and_populate(r'.*spots.*\.csv')

            if not validate_spots_df(merged_spots_df):
                print("Error: Validation failed for merged spots dataframe.")
            else:
                merged_spots_df = sort_and_generate_repeat(merged_spots_df)
                merged_spots_df['Unique_ID'] = merged_spots_df['File_name'] + "_" + merged_spots_df['TRACK_ID'].astype(str)
                merged_spots_df.dropna(subset=['POSITION_X', 'POSITION_Y', 'POSITION_Z'], inplace=True)
                save_dataframe_with_progress(
                    merged_spots_df,
                    os.path.join(Results_Folder, 'merged_Spots.csv'),
                    desc="Saving Spots"
                )
                # Now, call the check function
                check_unique_id_match(merged_spots_df, merged_tracks_df)

    if Folder_path:
        output.value = (
            "<br>✅ Done"
            f"<br><br><strong>Folder path:</strong> {Folder_path}"
            f"<br><strong>Results folder:</strong> {Results_Folder}"
        )
    else:
        output.value = (
            "<br>⚠️ No folder path provided."
            f"<br><br><strong>Results folder:</strong> {Results_Folder}"
        )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Provide the path to your dataset</h2>"),
    Folder_path_widget,
    widgets.HTML("<h3>Provide the path to your Result folder</h3>"),
    Results_Folder_widget,
    submit_button,
    process_output,
    output,
)

## **1.3. Mapping Wells to Conditions (optional)**
---

In this section of the notebook, conditions are defined based on user inputs, allowing for flexible and customized categorization of the experimental data.

*   When multiple wells are assigned the same condition, they are treated as repeats

*   Within each well, data from all fields of view are combined





In [ ]:
# @markdown ##Run to perform the mapping

def generate_repeat(group):
    unique_experiment_nbs = sorted(group['Well'].unique())
    experiment_nb_to_repeat = {experiment_nb: i+1 for i, experiment_nb in enumerate(unique_experiment_nbs)}
    group['Repeat'] = group['Well'].map(experiment_nb_to_repeat)
    return group

def sort_and_generate_repeat(merged_df):
    merged_df.sort_values(['Condition', 'Well'], inplace=True)
    merged_df = merged_df.groupby('Condition', group_keys=False).apply(generate_repeat)
    return merged_df

# Create a Text widget for each well
well_widgets = {well: widgets.Text(description=f"Well {well}") for well in merged_tracks_df['Well'].unique()}

# Function to update dataframe with the input conditions
def update_conditions(button):
    global merged_tracks_df  # Add this line to use the global variable

    for well, widget in well_widgets.items():
        condition = widget.value.strip()  # Trim whitespace from the input
        if condition:  # Only update if a condition was entered
            merged_tracks_df.loc[merged_tracks_df['Well'] == well, 'Condition'] = condition
    print("Conditions updated.")  # Feedback to confirm the update
    merged_tracks_df = sort_and_generate_repeat(merged_tracks_df)

    print("Repeats updated.")  # Feedback to confirm the update
    save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv', desc="Saving Tracks")
    check_for_nans(merged_tracks_df, "merged_tracks_df")

# Display the widgets
for widget in well_widgets.values():
    display(widget)

# Button to update the conditions
update_button = widgets.Button(description="Update Conditions")
update_button.on_click(update_conditions)
display(update_button)


--------------------------------------------------------
# **Part 2. Visualise your tracks**
--------------------------------------------------------

## **2.1 Visualise your tracks in each field of view**
---

In [ ]:
# @markdown ##Run the cell and choose the file you want to inspect
display_plots=True

if not os.path.exists(Results_Folder+"/Tracks"):
    os.makedirs(Results_Folder+"/Tracks")

filenames = merged_spots_df['File_name'].unique()

filename_dropdown = widgets.Dropdown(
    options=filenames,
    value=filenames[0] if len(filenames) > 0 else None,  # Default selected value
    description='File Name:',
)

interact(lambda filename: plot_track_coordinates(filename, merged_spots_df, Results_Folder, max_x, max_y, display_plots=display_plots), filename=filename_dropdown)


In [ ]:
#@markdown ##Run this cell to process all fields of view

# -----------------------------
# Widget UI
# -----------------------------

display_plots_widget = widgets.Checkbox(
    value=False,
    description="Display plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Process all fields of view",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global display_plots

    output.value = "🔁 Plotting and saving tracks for all FOVs ..."
    process_output.clear_output(wait=True)

    # Read widget values
    display_plots = display_plots_widget.value

    with process_output:
        for filename in tqdm(filenames, desc="Processing"):
            plot_track_coordinates(
                filename,
                merged_spots_df,
                Results_Folder,
                display_plots=display_plots
            )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>All plots saved in:</strong> {Results_Folder}/Tracks/"
        f"<br><strong>Display plots:</strong> {display_plots}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Process all fields of view</h2>"),
    display_plots_widget,
    submit_button,
    process_output,
    output,
)

## **2.2 Origin-Normalized Plot for each field of view**

In [ ]:
# @markdown ##Run the cell and choose the file you want to inspect

display_plots=True

if not os.path.exists(Results_Folder+"/Tracks"):
    os.makedirs(Results_Folder+"/Tracks")

filenames = merged_spots_df['File_name'].unique()

filename_dropdown = widgets.Dropdown(
    options=filenames,
    value=filenames[0] if len(filenames) > 0 else None,
    description='File Name:',
)

interact(lambda filename: plot_origin_normalized_coordinates_FOV(filename, merged_spots_df, Results_Folder), filename=filename_dropdown)


In [ ]:
#@markdown ##Run this cell to process all fields of view

# -----------------------------
# Widget UI
# -----------------------------

display_plots_widget = widgets.Checkbox(
    value=False,
    description="Display plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Process all fields of view",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global display_plots

    output.value = "🔁 Plotting and saving tracks for all FOVs ..."
    process_output.clear_output(wait=True)

    # Read widget values
    display_plots = display_plots_widget.value

    with process_output:
        for filename in tqdm(filenames, desc="Processing"):
            plot_origin_normalized_coordinates_FOV(
                filename,
                merged_spots_df,
                Results_Folder,
                display_plots=display_plots
            )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>All plots saved in:</strong> {Results_Folder}/Tracks/"
        f"<br><strong>Display plots:</strong> {display_plots}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Process all fields of view</h2>"),
    display_plots_widget,
    submit_button,
    process_output,
    output,
)

## **2.3 Origin-Normalized Plot for each condition and repeat**

In [ ]:
# @markdown ##Run the cell and choose the file you want to inspect

if not os.path.exists(Results_Folder + "/Tracks"):
    os.makedirs(Results_Folder + "/Tracks")  # Ensure the directory exists for saving the plots

conditions = merged_spots_df['Condition'].unique()
repeats = merged_spots_df['Repeat'].unique()

condition_dropdown = widgets.Dropdown(
    options=conditions,
    value=conditions[0] if len(conditions) > 0 else None,
    description='Condition:',
)

repeat_dropdown = widgets.Dropdown(
    options=repeats,
    value=repeats[0] if len(repeats) > 0 else None,
    description='Repeat:',
)

interact(lambda condition, repeat: plot_origin_normalized_coordinates_condition_repeat(
            condition, repeat, merged_spots_df, Results_Folder),
         condition=condition_dropdown,
         repeat=repeat_dropdown);

In [ ]:
#@markdown ##Run this cell to process all Repeat/Condition combinations

# -----------------------------
# Widget UI
# -----------------------------

display_plots_widget = widgets.Checkbox(
    value=False,
    description="Display plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Process all Repeat/Condition combinations",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global display_plots
    global conditions
    global repeats

    output.value = "🔁 Plotting and saving tracks for all combinations of Conditions and Repeats ..."
    process_output.clear_output(wait=True)

    # Read widget values
    display_plots = display_plots_widget.value

    # Ensure the directory exists for saving the plots
    if not os.path.exists(os.path.join(Results_Folder, "Tracks")):
        os.makedirs(os.path.join(Results_Folder, "Tracks"), exist_ok=True)

    conditions = merged_spots_df["Condition"].unique()
    repeats = merged_spots_df["Repeat"].unique()

    with process_output:
        for condition in tqdm(conditions, desc="Conditions"):
            for repeat in tqdm(repeats, desc="Repeats", leave=False):
                plot_origin_normalized_coordinates_condition_repeat(
                    condition,
                    repeat,
                    merged_spots_df,
                    Results_Folder,
                    display_plots=display_plots
                )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>All plots saved in:</strong> {Results_Folder}/Tracks/"
        f"<br><strong>Display plots:</strong> {display_plots}"
        f"<br><strong>Conditions processed:</strong> {len(conditions)}"
        f"<br><strong>Repeats processed:</strong> {len(repeats)}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Process all Repeat/Condition combinations</h2>"),
    display_plots_widget,
    submit_button,
    process_output,
    output,
)

## **2.4 Origin-Normalized Plot for each condition**

In [ ]:
# @markdown ##Run the cell and choose the file you want to inspect

if not os.path.exists(Results_Folder + "/Tracks"):
    os.makedirs(Results_Folder + "/Tracks")  # Ensure the directory exists for saving the plots

conditions = merged_spots_df['Condition'].unique()

condition_dropdown = widgets.Dropdown(
    options=conditions,
    value=conditions[0] if len(conditions) > 0 else None,
    description='Condition:',
)

interact(lambda condition: plot_origin_normalized_coordinates_condition(
            condition, merged_spots_df, Results_Folder),
         condition=condition_dropdown);

In [ ]:
#@markdown ##Run this cell to process all conditions

# -----------------------------
# Widget UI
# -----------------------------

display_plots_widget = widgets.Checkbox(
    value=False,
    description="Display plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Process all conditions",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global display_plots
    global conditions

    output.value = "🔁 Plotting and saving tracks for all Conditions ..."
    process_output.clear_output(wait=True)

    # Read widget values
    display_plots = display_plots_widget.value

    # Ensure the directory exists for saving the plots
    if not os.path.exists(os.path.join(Results_Folder, "Tracks")):
        os.makedirs(os.path.join(Results_Folder, "Tracks"), exist_ok=True)

    conditions = merged_spots_df["Condition"].unique()

    with process_output:
        for condition in tqdm(conditions, desc="Conditions"):
            plot_origin_normalized_coordinates_condition(
                condition,
                merged_spots_df,
                Results_Folder,
                display_plots=display_plots
            )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>All plots saved in:</strong> {Results_Folder}/Tracks/"
        f"<br><strong>Display plots:</strong> {display_plots}"
        f"<br><strong>Conditions processed:</strong> {len(conditions)}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Process all conditions</h2>"),
    display_plots_widget,
    submit_button,
    process_output,
    output,
)

## **2.5 Plot the migration vectors for each field of view**

In [ ]:
# @markdown ##Plot the migration vectors
display_plots=True

fovs = merged_spots_df['File_name'].unique()
fov_dropdown = Dropdown(
    options=fovs,
    value=fovs[0] if len(fovs) > 0 else None,
    description='Select FOV:',
)

interact(lambda filename, display_plots: plot_migration_vectors(filename, merged_spots_df, Results_Folder, display_plots),
         filename=fov_dropdown,
         display_plots=display_plots);

In [ ]:
#@markdown ##Run this cell to process all fields of view

# -----------------------------
# Widget UI
# -----------------------------

display_plots_widget = widgets.Checkbox(
    value=False,
    description="Display plots"
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Process all fields of view",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global display_plots

    output.value = "🔁 Plotting and saving track vectors for all FOVs ..."
    process_output.clear_output(wait=True)

    # Read widget values
    display_plots = display_plots_widget.value

    with process_output:
        for filename in tqdm(filenames, desc="Processing"):
            plot_migration_vectors(
                filename,
                merged_spots_df,
                Results_Folder,
                display_plots=display_plots
            )

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>All plots saved in:</strong> {Results_Folder}/Tracks/"
        f"<br><strong>Display plots:</strong> {display_plots}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Process all fields of view</h2>"),
    display_plots_widget,
    submit_button,
    process_output,
    output,
)

--------------------------------------------------------
# **Part 3. Compute additional metrics (optional)**
--------------------------------------------------------
<font color="red">Part2 does not support Track splitting</font>.

For users aiming to compute additional track metrics within this environment, it is crucial to disable track splitting in TrackMate.


## **3.1 Filter and smooth your tracks (Optional)**
---


The following section provides an interactive way to refine your tracking data. Here's what it's designed to achieve:

1. **Filter Tracks**:
    - This feature allows you to define a range for the track lengths you're interested in. By adjusting the `Min Length` and `Max Length` sliders, you can ignore very short or very long tracks that might be artifacts or noise in your data.

2. **Smooth Tracks**:
    - The positional data in your tracks can be smoothed using a moving average technique. By adjusting the `Smoothing` slider, you can control the degree of smoothing applied to the tracks. A higher value will average over more points, producing smoother tracks. This can be beneficial if your raw data has a lot of jitter or minor positional fluctuations.

**How to Use**:

- **Min Length**: Use the slider to set the minimum length of the tracks you're interested in.
- **Max Length**: Use the slider to set the maximum length of the tracks you're interested in.
- **Smoothing**: Adjust this slider to control the degree of smoothing you'd like to apply to your tracks.
- **Apply Filters**: After adjusting the sliders to your preference, click this button. This will process the data based on your choices and prepare it for downstream analyses.



In [ ]:
# @markdown ##Run to compute basic track metrics for filtering purpose

tqdm.pandas(desc="Calculating track metrics for filtering purpose")

global_metrics_df = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_track_metrics)

In [ ]:
# @markdown ##Run to filter and smooth your tracks (slow when the dataset is large)

duration_slider = create_metric_slider('Duration:', 'Track Duration', global_metrics_df, width='500px')
mean_speed_slider = create_metric_slider('Mean Speed:', 'Mean Speed', global_metrics_df, width='500px')
max_speed_slider = create_metric_slider('Max Speed:', 'Max Speed', global_metrics_df, width='500px')
min_speed_slider = create_metric_slider('Min Speed:', 'Min Speed', global_metrics_df, width='500px')
total_distance_slider = create_metric_slider('Total Distance:', 'Total Distance Traveled', global_metrics_df, width='500px')
smoothing_slider = widgets.IntSlider(
    value=3,  # Default value; adjust as needed
    min=1,    # Minimum value
    max=10,   # Maximum value, adjust based on expected maximum
    step=1,   # Step value for the slider
    description='Smoothing Neighbors:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')  # Adjust width to match other sliders if necessary
)

def filter_on_button_click(button):
    global filtered_and_smoothed_df
    metric_filters = {
        'Track Duration': duration_slider.value,
        'Mean Speed': mean_speed_slider.value,
        'Max Speed': max_speed_slider.value,
        'Min Speed': min_speed_slider.value,
        'Total Distance Traveled': total_distance_slider.value,
    }
    with output:
        clear_output(wait=True)
        filtered_and_smoothed_df, metrics_summary_df = optimized_filter_and_smooth_tracks(
            merged_spots_df,
            metric_filters,
            smoothing_neighbors=smoothing_slider.value,
            global_metrics_df=global_metrics_df
        )
        # Save parameters
        params_file_path = os.path.join(Results_Folder, "filter_smoothing_parameters.csv")
        save_filter_smoothing_params(
            params_file_path,
            smoothing_slider.value,
            duration_slider.value,
            mean_speed_slider.value,
            max_speed_slider.value,
            min_speed_slider.value,
            total_distance_slider.value
        )
        print("Filtering and Smoothing Done")

apply_button = widgets.Button(description="Apply Filters", button_style='info')
apply_button.on_click(filter_on_button_click)
output = widgets.Output()

display_widgets = widgets.VBox([
    smoothing_slider,
    duration_slider, mean_speed_slider, max_speed_slider, min_speed_slider, total_distance_slider,
    apply_button, output
])
display(display_widgets)

In [ ]:
# @markdown ##Compare Raw vs Filtered tracks

if not os.path.exists(Results_Folder+"/Tracks"):
    os.makedirs(Results_Folder+"/Tracks")  # Create Results_Folder if it doesn't exist

# Extract unique filenames from the dataframe
filenames = merged_spots_df['File_name'].unique()

# Create a Dropdown widget with the filenames
filename_dropdown = widgets.Dropdown(
    options=filenames,
    value=filenames[0] if len(filenames) > 0 else None,  # Default selected value
    description='File Name:',
)

# Link the Dropdown widget to the plotting function
interact(lambda filename: plot_coordinates_side_by_side(filename, merged_spots_df, filtered_and_smoothed_df, Results_Folder), filename=filename_dropdown);

In [ ]:
# @markdown ##Run to choose which data you want to use for further analysis

widget_layout = widgets.Layout(width='500px')

# Create a RadioButtons widget to allow users to choose the DataFrame
data_choice = widgets.RadioButtons(
    options=[('Raw data', 'raw'), ('Smooth and filtered data', 'smoothed')],
    description='Use:',
    value='raw',
    disabled=False,
    layout=widget_layout
)

# Create a button for analysis
analyze_button = widgets.Button(
    description="Analyze",
    button_style='info',
    layout=widget_layout
)

# Define the button click callback
def on_analyze_button_click(button):
    global spots_df_to_use
    global merged_tracks_df

    if data_choice.value == 'smoothed':
        merged_spots_df = filtered_and_smoothed_df
        save_dataframe_with_progress(merged_spots_df, Results_Folder + '/' + 'merged_Spots.csv')
        merged_tracks_df = merged_tracks_df[merged_tracks_df['Unique_ID'].isin(merged_spots_df['Unique_ID'])]
        save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

    print(f"Analysis will be performed using: {data_choice.label}")

# Assign button callback
analyze_button.on_click(on_analyze_button_click)

# Initial display of the widgets
display(data_choice)
display(analyze_button)


## **3.2. Duration and speed metrics**
---
When this cell is executed, it calculates various metrics for each unique track. Specifically, for each track, it determines the duration of the track, the average, maximum, minimum, and standard deviation of speeds, as well as the total distance traveled by the tracked object.

In [ ]:
# @markdown ##Calculate track metrics

print("Calculating track metrics...")

merged_spots_df.dropna(subset=['POSITION_X', 'POSITION_Y', 'POSITION_Z'], inplace=True)

tqdm.pandas(desc="Calculating Track Metrics")

columns_to_remove = [
    "TRACK_DURATION",
    "TRACK_MEAN_SPEED",
    "TRACK_MAX_SPEED",
    "TRACK_MIN_SPEED",
    "TRACK_MEDIAN_SPEED",
    "TRACK_STD_SPEED",
    "TOTAL_DISTANCE_TRAVELED"
]

for column in columns_to_remove:
    if column in merged_tracks_df.columns:
        merged_tracks_df.drop(column, axis=1, inplace=True)

merged_spots_df.sort_values(by=['Unique_ID', 'POSITION_T'], inplace=True)
df_track_metrics = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_track_metrics).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_track_metrics.columns).drop('Unique_ID')
merged_tracks_df.drop(columns=overlapping_columns, inplace=True)
merged_tracks_df = pd.merge(merged_tracks_df, df_track_metrics, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')
check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

In [ ]:
#@markdown ##Run this cell to calculate track metrics using rolling windows

# -----------------------------
# Widget UI
# -----------------------------

window_size_widget = widgets.IntText(
    value=5,
    description="Window size:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Calculate rolling-window track metrics",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global window_size
    global df_track_metrics
    global merged_tracks_df

    output.value = "🔁 Calculating track metrics using a rolling window ..."
    process_output.clear_output(wait=True)

    # Read widget values
    window_size = window_size_widget.value

    with process_output:
        tqdm.pandas(desc="Calculating Track Metrics using a rolling window")

        merged_spots_df.sort_values(
            by=["Unique_ID", "POSITION_T"],
            inplace=True
        )

        df_track_metrics = (
            merged_spots_df
            .groupby("Unique_ID")
            .progress_apply(
                lambda x: calculate_track_metrics_rolling(
                    x,
                    window_size=window_size
                )
            )
            .reset_index()
        )

        overlapping_columns = (
            merged_tracks_df
            .columns
            .intersection(df_track_metrics.columns)
            .drop("Unique_ID")
        )

        merged_tracks_df.drop(
            columns=overlapping_columns,
            inplace=True
        )

        merged_tracks_df = pd.merge(
            merged_tracks_df,
            df_track_metrics,
            on="Unique_ID",
            how="left"
        )

        save_dataframe_with_progress(
            merged_tracks_df,
            os.path.join(Results_Folder, "merged_Tracks.csv")
        )

        check_for_nans(merged_tracks_df, "merged_tracks_df")

    output.value = (
        "<br>✅ Done"
        f"<br><br><strong>Window size:</strong> {window_size}"
        f"<br><strong>Saved file:</strong> {os.path.join(Results_Folder, 'merged_Tracks.csv')}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Calculate track metrics using rolling windows</h2>"),
    window_size_widget,
    submit_button,
    process_output,
    output,
)

## **3.3. Directionality**
---
To calculate the directionality of a track in 3D space, we consider a series of points each with \(x\), \(y\), and \(z\) coordinates, sorted by time. The directionality, denoted as \(D\), is calculated using the formula:

$$ D = \frac{d_{\text{euclidean}}}{d_{\text{total path}}} $$

where \($d_{\text{euclidean}}$\) is the Euclidean distance between the first and the last points of the track, calculated as:

$$ d_{\text{euclidean}} = \sqrt{(x_{\text{end}} - x_{\text{start}})^2 + (y_{\text{end}} - y_{\text{start}})^2 + (z_{\text{end}} - z_{\text{start}})^2} $$

and \($d_{\text{total path}}$\) is the sum of the Euclidean distances between all consecutive points in the track, representing the total path length traveled. If the total path length is zero, the directionality is defined to be zero. This measure provides insight into the straightness of the path taken, with a value of 1 indicating a straight path between the start and end points, and values approaching 0 indicating more circuitous paths.</font>


In [ ]:
# @markdown ##Calculate directionality
from celltracks.Track_Metrics import calculate_directionality

print("In progress...")

merged_spots_df.dropna(subset=['POSITION_X', 'POSITION_Y', 'POSITION_Z'], inplace=True)

tqdm.pandas(desc="Calculating Directionality")

merged_spots_df.sort_values(by=['Unique_ID', 'POSITION_T'], inplace=True)

df_directionality = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_directionality).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_directionality.columns).drop('Unique_ID')

merged_tracks_df.drop(columns=overlapping_columns, inplace=True)

merged_tracks_df = pd.merge(merged_tracks_df, df_directionality, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

In [ ]:
#@markdown ##Run this cell to calculate directionality using rolling windows

# -----------------------------
# Widget UI
# -----------------------------

window_size_widget = widgets.IntText(
    value=5,
    description="Window size:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Calculate rolling directionality",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global window_size
    global df_rolling_directionality
    global merged_tracks_df

    output.value = "🔁 Calculating rolling directionality ..."
    process_output.clear_output(wait=True)

    # Read widget values
    window_size = window_size_widget.value

    with process_output:
        tqdm.pandas(desc="Calculating Rolling Directionality")

        df_rolling_directionality = (
            merged_spots_df
            .groupby("Unique_ID")
            .progress_apply(
                lambda x: calculate_rolling_directionality(
                    x,
                    window_size=window_size
                )
            )
            .reset_index()
        )

        overlapping_columns = (
            merged_tracks_df
            .columns
            .intersection(df_rolling_directionality.columns)
            .drop("Unique_ID")
        )

        merged_tracks_df.drop(
            columns=overlapping_columns,
            inplace=True
        )

        merged_tracks_df = pd.merge(
            merged_tracks_df,
            df_rolling_directionality,
            on="Unique_ID",
            how="left"
        )

        save_dataframe_with_progress(
            merged_tracks_df,
            os.path.join(Results_Folder, "merged_Tracks.csv")
        )

        check_for_nans(merged_tracks_df, "merged_tracks_df")

    output.value = (
        "<br>✅ Rolling Directionality Calculation...Done"
        f"<br><br><strong>Window size:</strong> {window_size}"
        f"<br><strong>Saved file:</strong> {os.path.join(Results_Folder, 'merged_Tracks.csv')}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Calculate directionality using rolling windows</h2>"),
    window_size_widget,
    submit_button,
    process_output,
    output,
)

## **3.4. Tortuosity**
---
This measure provides insight into the curvature and complexity of the path taken, with a value of 1 indicating a straight path between the start and end points, and values greater than 1 indicating paths with more twists and turns.
To calculate the tortuosity of a track in 3D space, we consider a series of points each with \(x\), \(y\), and \(z\) coordinates, sorted by time. The tortuosity, denoted as \(T\), is calculated using the formula:

$$ T = \frac{d_{\text{total path}}}{d_{\text{euclidean}}} $$



In [ ]:
# @markdown ##Calculate tortuosity
print("In progress...")

tqdm.pandas(desc="Calculating Tortuosity")

merged_spots_df.sort_values(by=['Unique_ID', 'POSITION_T'], inplace=True)

df_tortuosity = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_tortuosity).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_tortuosity.columns).drop('Unique_ID')

merged_tracks_df.drop(columns=overlapping_columns, inplace=True)

merged_tracks_df = pd.merge(merged_tracks_df, df_tortuosity, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

In [ ]:
#@markdown ##Run this cell to calculate tortuosity using rolling windows

# -----------------------------
# Widget UI
# -----------------------------

window_size_widget = widgets.IntText(
    value=5,
    description="Window size:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Calculate rolling tortuosity",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global window_size
    global df_rolling_tortuosity
    global merged_tracks_df

    output.value = "🔁 Calculating rolling tortuosity ..."
    process_output.clear_output(wait=True)

    # Read widget values
    window_size = window_size_widget.value

    with process_output:
        tqdm.pandas(desc="Calculating Rolling Tortuosity")

        df_rolling_tortuosity = (
            merged_spots_df
            .groupby("Unique_ID")
            .progress_apply(
                lambda x: calculate_rolling_tortuosity(
                    x,
                    window_size=window_size
                )
            )
            .reset_index()
        )

        overlapping_columns = (
            merged_tracks_df
            .columns
            .intersection(df_rolling_tortuosity.columns)
            .drop("Unique_ID")
        )

        merged_tracks_df.drop(
            columns=overlapping_columns,
            inplace=True
        )

        merged_tracks_df = pd.merge(
            merged_tracks_df,
            df_rolling_tortuosity,
            on="Unique_ID",
            how="left"
        )

        save_dataframe_with_progress(
            merged_tracks_df,
            os.path.join(Results_Folder, "merged_Tracks.csv")
        )

        check_for_nans(merged_tracks_df, "merged_tracks_df")

    output.value = (
        "<br>✅ Rolling Tortuosity Calculation...Done"
        f"<br><br><strong>Window size:</strong> {window_size}"
        f"<br><strong>Saved file:</strong> {os.path.join(Results_Folder, 'merged_Tracks.csv')}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Calculate tortuosity using rolling windows</h2>"),
    window_size_widget,
    submit_button,
    process_output,
    output,
)

## **3.5. Calculate the total turning angle**
---

This measure provides insight into the cumulative amount of turning along the path, with a value of 0 indicating a straight path with no turning, and higher values indicating paths with more turning.

To calculate the Total Turning Angle of a track in 3D space, we consider a series of points each with \(x\), \(y\), and \(z\) coordinates, sorted by time. The Total Turning Angle, denoted as \(A\), is the sum of the angles between each pair of consecutive direction vectors along the track, representing the cumulative amount of turning along the path.

For each pair of consecutive segments in the track, we calculate the direction vectors \( $\vec{v_1}$ \) and \($ \vec{v_2}$ \), and the angle \($ \theta$ \) between them is calculated using the formula:

$$ \cos(\theta) = \frac{\vec{v_1} \cdot \vec{v_2}}{||\vec{v_1}|| \cdot ||\vec{v_2}||} $$

where \( $\vec{v_1} \cdot$ $\vec{v_2}$ \) is the dot product of the direction vectors, and \( $||\vec{v_1}||$ \) and \( $||\vec{v_2}||$ \) are the magnitudes of the direction vectors. The Total Turning Angle \( $A$ \) is then the sum of all the angles \( \$theta$ \) calculated between each pair of consecutive direction vectors along the track:

$$ A = \sum \theta $$

If either of the direction vectors is a zero vector, the angle between them is undefined, and such cases are skipped in the calculation.


In [ ]:
# @markdown ##Calculate the total turning angle

tqdm.pandas(desc="Calculating Total Turning Angle")

merged_spots_df.sort_values(by=['Unique_ID', 'POSITION_T'], inplace=True)

df_turning_angle = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_total_turning_angle).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_turning_angle.columns).drop('Unique_ID')

merged_tracks_df.drop(columns=overlapping_columns, inplace=True)

merged_tracks_df = pd.merge(merged_tracks_df, df_turning_angle, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

In [ ]:
#@markdown ##Run this cell to calculate the total turning angle using rolling windows

# -----------------------------
# Widget UI
# -----------------------------

window_size_widget = widgets.IntText(
    value=5,
    description="Window size:",
    style={"description_width": "120px"}
)

submit_button = widgets.Button(
    description="Submit",
    button_style="success",
    tooltip="Calculate rolling total turning angle",
    icon="check"
)

output = widgets.HTML()
process_output = widgets.Output()

# -----------------------------
# Submit button logic
# -----------------------------

def on_submit_clicked(button):
    global window_size
    global df_rolling_turning_angle
    global merged_tracks_df

    output.value = "🔁 Calculating rolling total turning angle ..."
    process_output.clear_output(wait=True)

    # Read widget values
    window_size = window_size_widget.value

    with process_output:
        tqdm.pandas(desc="Calculating Average Total Turning Angle")

        df_rolling_turning_angle = (
            merged_spots_df
            .groupby("Unique_ID")
            .progress_apply(
                lambda x: calculate_rolling_total_turning_angle(
                    x,
                    window_size=window_size
                )
            )
            .reset_index()
        )

        overlapping_columns = (
            merged_tracks_df
            .columns
            .intersection(df_rolling_turning_angle.columns)
            .drop("Unique_ID")
        )

        merged_tracks_df.drop(
            columns=overlapping_columns,
            inplace=True
        )

        merged_tracks_df = pd.merge(
            merged_tracks_df,
            df_rolling_turning_angle,
            on="Unique_ID",
            how="left"
        )

        save_dataframe_with_progress(
            merged_tracks_df,
            os.path.join(Results_Folder, "merged_Tracks.csv")
        )

        check_for_nans(merged_tracks_df, "merged_tracks_df")

    output.value = (
        "<br>✅ Rolling Total Turning Angle Calculation...Done"
        f"<br><br><strong>Window size:</strong> {window_size}"
        f"<br><strong>Saved file:</strong> {os.path.join(Results_Folder, 'merged_Tracks.csv')}"
    )

submit_button.on_click(on_submit_clicked)

display(
    widgets.HTML("<h2>Calculate the total turning angle using rolling windows</h2>"),
    window_size_widget,
    submit_button,
    process_output,
    output,
)

## **3.6. Calculate the Spatial Coverage**
---

Spatial coverage provides insight into the spatial extent covered by the object's movement, with higher values indicating that the object has covered a larger area or volume during its movement.


To calculate the spatial coverage of a track in 2D or 3D space, we consider a series of points each with \(x\), \(y\), and optionally \(z\) coordinates, sorted by time. The spatial coverage, denoted as \(S\), represents the area (in 2D) or volume (in 3D) enclosed by the convex hull formed by the points in the track. It provides insight into the spatial extent covered by the moving object.

In the implementation below we:
1. **Check Dimensionality**:
   - If the variance of the \(z\) coordinates is zero, implying all \(z\) coordinates are the same, the spatial coverage is calculated in 2D using only the \(x\) and \(y\) coordinates.
   - If the \(z\) coordinates vary, the spatial coverage is calculated in 3D using the \(x\), \(y\), and \(z\) coordinates.

2. **Form Convex Hull**:
   - In 2D, a minimum of 3 non-collinear points is required to form a convex hull.
   - In 3D, a minimum of 4 non-coplanar points is required to form a convex hull.
   - If the required minimum points are not available, the spatial coverage is defined to be zero.

3. **Calculate Spatial Coverage**:
   - In 2D, the spatial coverage \(S\) is the area of the convex hull formed by the points in the track.
   - In 3D, the spatial coverage \(S\) is the volume of the convex hull formed by the points in the track.

Formula:
- For 2D Spatial Coverage (Area of Convex Hull), if points are \(P_1(x_1, y_1), P_2(x_2, y_2), \ldots, P_n(x_n, y_n)\):
  $$ S_{2D} = \text{Area of Convex Hull formed by } P_1, P_2, \ldots, P_n $$

- For 3D Spatial Coverage (Volume of Convex Hull), if points are \(P_1(x_1, y_1, z_1), P_2(x_2, y_2, z_2), \ldots, P_n(x_n, y_n, z_n)\):
  $$ S_{3D} = \text{Volume of Convex Hull formed by } P_1, P_2, \ldots, P_n $$



In [ ]:
# @markdown ##Calculate the Spatial Coverage

tqdm.pandas(desc="Calculating Spatial Coverage")

merged_spots_df.sort_values(by=['Unique_ID', 'POSITION_T'], inplace=True)

df_spatial_coverage = merged_spots_df.groupby('Unique_ID').progress_apply(calculate_spatial_coverage).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_spatial_coverage.columns).drop('Unique_ID')

merged_tracks_df.drop(columns=overlapping_columns, inplace=True)

merged_tracks_df = pd.merge(merged_tracks_df, df_spatial_coverage, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

In [ ]:
# @markdown ##Calculate the Spatial Coverage using rolling windows

window_size = 5  # Adjust as needed

tqdm.pandas(desc="Calculating Rolling Spatial Coverage")

df_rolling_spatial_coverage = merged_spots_df.groupby('Unique_ID').progress_apply(lambda x: calculate_rolling_spatial_coverage(x, window_size=window_size)).reset_index()

overlapping_columns = merged_tracks_df.columns.intersection(df_rolling_spatial_coverage.columns).drop('Unique_ID')
merged_tracks_df.drop(columns=overlapping_columns, inplace=True)

merged_tracks_df = pd.merge(merged_tracks_df, df_rolling_spatial_coverage, on='Unique_ID', how='left')

save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("Rolling Spatial Coverage Calculation...Done")

## **3.7. Compute additional metrics**
---

This cell computes various metrics for each track in the provided dataset. These metrics are derived from the information provided by TrackMate in the spots table and include statistical properties like mean, median, standard deviation, minimum, and maximum values. For further information about these metrics visit the corresponding [TrackMate](https://imagej.net/plugins/trackmate/analyzers/#spot-analyzers).


In [ ]:
# @markdown ##Compute additional metrics

print("In progress...")

# List of potential metrics to compute
potential_metrics = [
    'MEAN_INTENSITY_CH1', 'MEDIAN_INTENSITY_CH1', 'MIN_INTENSITY_CH1', 'MAX_INTENSITY_CH1',
    'TOTAL_INTENSITY_CH1', 'STD_INTENSITY_CH1', 'CONTRAST_CH1', 'SNR_CH1', 'ELLIPSE_X0',
    'ELLIPSE_Y0', 'ELLIPSE_MAJOR', 'ELLIPSE_MINOR', 'ELLIPSE_THETA', 'ELLIPSE_ASPECTRATIO',
    'AREA', 'PERIMETER', 'CIRCULARITY', 'SOLIDITY', 'SHAPE_INDEX','MEAN_INTENSITY_CH2', 'MEDIAN_INTENSITY_CH2', 'MIN_INTENSITY_CH2', 'MAX_INTENSITY_CH2',
    'TOTAL_INTENSITY_CH2', 'STD_INTENSITY_CH2', 'CONTRAST_CH2', 'SNR_CH2', 'MEAN_INTENSITY_CH3', 'MEDIAN_INTENSITY_CH3', 'MIN_INTENSITY_CH3', 'MAX_INTENSITY_CH3',
    'TOTAL_INTENSITY_CH3', 'STD_INTENSITY_CH3', 'CONTRAST_CH3', 'SNR_CH3', 'MEAN_INTENSITY_CH4', 'MEDIAN_INTENSITY_CH4', 'MIN_INTENSITY_CH4', 'MAX_INTENSITY_CH4',
    'TOTAL_INTENSITY_CH4', 'STD_INTENSITY_CH4', 'CONTRAST_CH4', 'SNR_CH4'
]

available_metrics = check_metrics_availability(merged_spots_df, potential_metrics)

morphological_metrics_df = compute_morphological_metrics(merged_spots_df, available_metrics)

morphological_metrics_df.reset_index(inplace=True)

if 'Unique_ID' in merged_tracks_df.columns:
    overlapping_columns = merged_tracks_df.columns.intersection(morphological_metrics_df.columns).drop('Unique_ID', errors='ignore')
    merged_tracks_df.drop(columns=overlapping_columns, inplace=True)
    merged_tracks_df = merged_tracks_df.merge(morphological_metrics_df, on='Unique_ID', how='left')
    save_dataframe_with_progress(merged_tracks_df, Results_Folder + '/' + 'merged_Tracks.csv')

else:
    print("Error: 'Unique_ID' column missing in merged_tracks_df. Skipping merging with morphological metrics.")

check_for_nans(merged_tracks_df, "merged_tracks_df")

print("...Done")

--------
# **Part 4. Quality Control**
--------

      



## **4.1. Assess if your dataset is balanced**
---

In cell tracking and similar biological analyses, the balance of the dataset is important, particularly in ensuring that each biological repeat carries equal weight. Here's why this balance is essential:

Accurate Representation of Biological Variability

- **Capturing True Biological Variation**: Biological repeats are crucial for capturing the natural variability inherent in biological systems. Equal weighting ensures that this variability is accurately represented.
- **Reducing Sampling Bias**: By balancing the dataset, we avoid overemphasizing the characteristics of any single repeat, which might not be representative of the broader biological context.

If your data is too imbalanced, it may be useful to ensure that this does not shift your results.



In [ ]:
# @markdown ##Check the number of track per condition per repeats

if not os.path.exists(f"{Results_Folder}/QC"):
    os.makedirs(f"{Results_Folder}/QC")

result_df = count_tracks_by_condition_and_repeat(merged_tracks_df, f"{Results_Folder}/QC")


## **4.2. Compute Similarity Metrics between Field of Views (FOV) and between Conditions and Repeats**
---

**Purpose**:

This section provides a set of tools to compute and visualize similarities between different field of views (FOV) based on selected track parameters. By leveraging hierarchical clustering, the resulting dendrogram offers a clear visualization of how different FOV, conditions, or repeats relate to one another. This tool is essential for:

1. **Quality Control**:
    - Ensuring that FOVs from the same condition or experimental setup are more similar to each other than to FOVs from different conditions.
    - Confirming that repeats of the same experiment yield consistent results and cluster together.
    
2. **Data Integrity**:
    - Identifying potential outliers or anomalies in the dataset.
    - Assessing the overall consistency of the experiment and ensuring reproducibility.

**How to Use**:

1. **Track Parameters Selection**:
    - A list of checkboxes allows users to select which track parameters they want to consider for similarity calculations. By default, all parameters are selected. Users can deselect parameters that they believe might not contribute significantly to the similarity.

2. **Similarity Metric**:
    - Users can choose a similarity metric from a dropdown list. Options include cosine, euclidean, cityblock, jaccard, and correlation. The choice of similarity metric can influence the clustering results, so users might need to experiment with different metrics to see which one provides the most meaningful results.

3. **Linkage Method**:
    - Determines how the distance between clusters is calculated in the hierarchical clustering process. Different linkage methods can produce different dendrograms, so users might want to try various methods.

4. **Visualization**:
    - Once the parameters are selected, users can click on the "Select the track parameters and visualize similarity" button. This will compute the hierarchical clustering and display two dendrograms:
        - One dendrogram displays similarities between individual FOVs.
        - Another dendrogram aggregates the data based on conditions and repeats, providing a higher-level view of the similarities.
      


In [ ]:
# @markdown ##Compute similarity metrics between FOV and between conditions and repeats

# Check and create "QC" folder
if not os.path.exists(f"{Results_Folder}/QC"):
    os.makedirs(f"{Results_Folder}/QC")

# Columns to exclude
excluded_columns = ['Condition', 'experiment_nb', 'File_name', 'Repeat', 'Unique_ID', 'LABEL', 'TRACK_INDEX', 'TRACK_ID', 'TRACK_X_LOCATION', 'TRACK_Y_LOCATION', 'TRACK_Z_LOCATION', 'Exemplar','TRACK_STOP', 'TRACK_START', 'Cluster_UMAP', 'Cluster_tsne']

selected_df = pd.DataFrame()

# Filter out non-numeric columns but keep 'File_name'
numeric_df = merged_tracks_df.select_dtypes(include=['float64', 'int64']).copy()
numeric_df['File_name'] = merged_tracks_df['File_name']

# Create a list of column names excluding 'File_name'
column_names = [col for col in numeric_df.columns if col not in excluded_columns]

# Create a checkbox for each column
checkboxes = [widgets.Checkbox(value=True, description=col, indent=False) for col in column_names]

# Dropdown for similarity metrics
similarity_dropdown = widgets.Dropdown(
    options=['cosine', 'euclidean', 'cityblock', 'jaccard', 'correlation'],
    value='cosine',
    description='Similarity Metric:'
)

# Dropdown for linkage methods
linkage_dropdown = widgets.Dropdown(
    options=['single', 'complete', 'average', 'ward'],
    value='single',
    description='Linkage Method:'
)

# Arrange checkboxes in a 2x grid
grid = widgets.GridBox(checkboxes, layout=widgets.Layout(grid_template_columns="repeat(2, 300px)"))

# Create a button to trigger the selection and visualization
button = widgets.Button(description="Select the track parameters and visualize similarity", layout=widgets.Layout(width='400px'), button_style='info')

# Define the button click event handler
def on_button_click(b):
    global selected_df  # Declare selected_df as global

    # Get the selected columns from the checkboxes
    selected_columns = [box.description for box in checkboxes if box.value]
    selected_columns.append('File_name')  # Always include 'File_name'

    # Extract the selected columns from the DataFrame
    selected_df = numeric_df[selected_columns]

    # Check and print the percentage of NaNs for each selected column
    for column in selected_columns:
        if selected_df[column].isna().any():
            nan_percentage = selected_df[column].isna().mean() * 100
            print("Warning: NaN values found in the selected data.")
            print(f"{column}: {nan_percentage:.2f}%")
            any_nan = True
            print("Proceeding to handle NaN values.")
            selected_df = selected_df.dropna()

    # Aggregate the data by filename
    aggregated_by_filename = selected_df.groupby('File_name').mean(numeric_only=True)

    # Aggregate the data by condition and repeat
    aggregated_by_condition_repeat = merged_tracks_df.groupby(['Condition', 'Repeat'])[selected_columns].mean(numeric_only=True)

    # Compute condensed distance matrices
    distance_matrix_filename = pdist(aggregated_by_filename, metric=similarity_dropdown.value)
    distance_matrix_condition_repeat = pdist(aggregated_by_condition_repeat, metric=similarity_dropdown.value)

    # Perform hierarchical clustering
    linked_filename = linkage(distance_matrix_filename, method=linkage_dropdown.value)
    linked_condition_repeat = linkage(distance_matrix_condition_repeat, method=linkage_dropdown.value)

    annotation_text = f"Similarity Method: {similarity_dropdown.value}, Linkage Method: {linkage_dropdown.value}"

        # Prepare the parameters dictionary
    similarity_params = {
        'Similarity Metric': similarity_dropdown.value,
        'Linkage Method': linkage_dropdown.value,
        'Selected Columns': ', '.join(selected_columns)
    }

    # Save the parameters
    params_file_path = os.path.join(Results_Folder, "QC/analysis_parameters.csv")
    save_parameters(similarity_params, params_file_path, 'Similarity Metrics')

    # Plot the dendrograms one under the other
    plt.figure(figsize=(10, 10))

    # Dendrogram for individual filenames
    plt.subplot(2, 1, 1)
    dendrogram(linked_filename, labels=aggregated_by_filename.index, orientation='top', distance_sort='descending', leaf_rotation=90)
    plt.title(f'Dendrogram of Field of view Similarities\n{annotation_text}')

    # Dendrogram for aggregated data based on condition and repeat
    plt.subplot(2, 1, 2)
    dendrogram(linked_condition_repeat, labels=aggregated_by_condition_repeat.index, orientation='top', distance_sort='descending', leaf_rotation=90)
    plt.title(f'Dendrogram of Aggregated Similarities by Condition and Repeat\n{annotation_text}')

    plt.tight_layout()

    # Save the dendrogram to a PDF
    pdf_pages = PdfPages(f"{Results_Folder}/QC/Dendrogram_Similarities.pdf")

    # Save the current figure to the PDF
    pdf_pages.savefig()

    # Close the PdfPages object to finalize the document
    pdf_pages.close()

    plt.show()

# Set the button click event handler
button.on_click(on_button_click)

# Display the widgets
display(grid, similarity_dropdown, linkage_dropdown, button)


-------------------------------------------

# **Part 5. Plot track parameters**
-------------------------------------------



 In this section you can plot all the track parameters previously computed. Data and graphs are automatically saved in your result folder.

<font color="red"> Parameters computed are in the unit you provided when tracking your data in TrackMate.</font>

<font size = 5>**Statistical analyses**

**Cohen's d (Effect Size):**

Cohen's d measures the size of the difference between two groups, normalized by their pooled standard deviation. Values can be interpreted as small (0 to 0.2), medium (0.2 to 0.5), or large (0.5 and above) effects. It helps quantify how significant the observed difference is, beyond just being statistically significant.

**Randomization Test:**

This non-parametric test evaluates if observed differences between conditions could have arisen by random chance. It shuffles condition labels multiple times, recalculating the Cohen's d each time. The resulting p-value, which indicates the likelihood of observing the actual difference by chance, provides evidence against the null hypothesis: a smaller p-value implies stronger evidence against the null.

**Bonferroni Correction:**

Given multiple comparisons, the Bonferroni Correction adjusts significance thresholds to mitigate the risk of false positives. By dividing the standard significance level (alpha) by the number of tests, it ensures that only robust findings are considered significant. However, it's worth noting that this method can be conservative, sometimes overlooking genuine effects.

## **5.1. Plot your entire dataset**
--------

In [ ]:
# @markdown ##Plot track normalized track parameters based on conditions as an heatmap (entire dataset)

base_folder = f"{Results_Folder}/track_parameters_plots"
Conditions = 'Condition'
df_to_plot = merged_tracks_df

folders = ["pdf", "csv"]
for folder in folders:
    dir_path = os.path.join(base_folder, folder)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

# Example usage
heatmap_comparison(merged_tracks_df, base_folder, Conditions, normalization="zscore")

In [ ]:
# @markdown ##Plot track parameters (entire dataset)

base_folder = f"{Results_Folder}/track_parameters_plots"
Conditions = 'Condition'
df_to_plot = merged_tracks_df

folders = ["pdf", "csv"]
for folder in folders:
    dir_path = os.path.join(base_folder, folder)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

condition_selector, condition_accordion = display_condition_selection(df_to_plot, Conditions)
checkboxes_dict, checkboxes_accordion = display_variable_checkboxes(categorize_columns(df_to_plot))
variable_checkboxes, checkboxes_widget = display_variable_checkboxes(get_selectable_columns_plots(df_to_plot))
stat_method_selector = widgets.Dropdown(
    options=['randomization test', 't-test'],
    value='randomization test',
    description='Stat Method:',
    style={'description_width': 'initial'}
)

button = Button(description="Plot Selected Variables", layout=Layout(width='400px'), button_style='info')
button.on_click(lambda b: plot_selected_vars(b, checkboxes_dict, df_to_plot, Conditions, base_folder, condition_selector, stat_method_selector));

display(VBox([condition_accordion, checkboxes_accordion, stat_method_selector, button]))

## **5.2. Plot a balanced dataset**
--------

### **5.2.1. Downsample your dataset to ensure that it is balanced**
--------

Downsampling and Balancing Dataset

This section of the notebook is dedicated to addressing imbalances in the dataset, which is crucial for ensuring the accuracy and reliability of the analysis. The cell bellow will downsample the dataset to balance the number of tracks across different conditions and repeats. It allows for reproducibility by including a `random_seed` parameter, which is set to 42 by default but can be adjusted as needed.

All results from this section will be saved in the Balanced Dataset Directory created in your `Results_Folder`.




In [ ]:
# @markdown ##Run this cell to downsample and balance your dataset

random_seed = 42

if not os.path.exists(f"{Results_Folder}/Balanced_dataset"):
    os.makedirs(f"{Results_Folder}/Balanced_dataset")

balanced_merged_tracks_df = balance_dataset(merged_tracks_df, random_seed=random_seed)
result_df = count_tracks_by_condition_and_repeat(balanced_merged_tracks_df, f"{Results_Folder}/Balanced_dataset")

check_for_nans(balanced_merged_tracks_df, "balanced_merged_tracks_df")
save_dataframe_with_progress(balanced_merged_tracks_df, Results_Folder + '/Balanced_dataset/merged_Tracks_balanced_dataset.csv')


### **5.2.2. Check if the downsampling has affected data distribution**
--------

This section of the notebook generates a heatmap visualizing the Kolmogorov-Smirnov (KS) p-values for each numerical column in the dataset, comparing the distributions before and after downsampling. This heatmap serves as a tool for assessing the impact of downsampling on data quality, guiding decisions on whether the downsampled dataset is suitable for further analysis.

Purpose of the Heatmap
- **KS Test:** The KS test is used to determine if two samples are drawn from the same distribution. In this context, it compares the distribution of each numerical column in the original dataset (`merged_tracks_df`) with its counterpart in the downsampled dataset (`balanced_merged_tracks_df`).
- **P-Value Interpretation:** The p-value indicates the probability that the two samples come from the same distribution. A higher p-value suggests a greater likelihood that the distributions are similar.

Interpreting the Heatmap
- **Color Coding:** The heatmap uses a color gradient (from viridis) to represent the range of p-values. Darker colors indicate higher p-values.
- **P-Value Thresholds:**
  - **High P-Values (Lighter Areas):** Indicate that the downsampling process likely did not significantly alter the distribution of that numerical column for the specific condition-repeat group.
  - **Low P-Values (Darker Areas):** Suggest that the downsampling process may have affected the distribution significantly.
- **Varying P-Values:** Variations in color across different columns and rows help identify which specific numerical columns and condition-repeat groups are most affected by the downsampling.




In [ ]:
# @markdown ##Check if your downsampling has affected your data distribution

numerical_columns = merged_tracks_df.select_dtypes(include=['int64', 'float64']).columns

# Initialize a DataFrame to store KS p-values
ks_p_values = pd.DataFrame(columns=numerical_columns)

# Iterate over each group and numerical column
for group, group_df in merged_tracks_df.groupby(['Condition', 'Repeat']):
    group_p_values = []
    balanced_group_df = balanced_merged_tracks_df[(balanced_merged_tracks_df['Condition'] == group[0]) & (balanced_merged_tracks_df['Repeat'] == group[1])]
    for column in numerical_columns:
        p_value = calculate_ks_p_value(group_df, balanced_group_df, column)
        group_p_values.append(p_value)
    ks_p_values.loc[f'Condition: {group[0]}, Repeat: {group[1]}'] = group_p_values

max_columns_per_heatmap = 20

total_columns = len(ks_p_values.columns)

num_heatmaps = -(-total_columns // max_columns_per_heatmap)  # Ceiling division

pdf_filepath = Results_Folder+'/Balanced_dataset/p-Value Heatmap.pdf'

# Create a PDF file
with PdfPages(pdf_filepath) as pdf:
    # Loop through each subset of columns and create a heatmap
    for i in range(num_heatmaps):
        start_col = i * max_columns_per_heatmap
        end_col = min(start_col + max_columns_per_heatmap, total_columns)

        # Subset of columns for this heatmap
        subset_columns = ks_p_values.columns[start_col:end_col]

        # Create the heatmap for the subset of columns
        plt.figure(figsize=(12, 8))
        sns.heatmap(ks_p_values[subset_columns], cmap='viridis', vmax=0.5, vmin=0)
        plt.title(f'Kolmogorov-Smirnov P-Value Heatmap (Columns {start_col+1} to {end_col})')
        plt.xlabel('Numerical Columns')
        plt.ylabel('Condition-Repeat Groups')
        plt.tight_layout()

        # Save the current figure to the PDF
        pdf.savefig()
        plt.show()
        plt.close()

print(f"Saved all heatmaps to {pdf_filepath}")

ks_p_values.to_csv(Results_Folder + '/Balanced_dataset/ks_p_values.csv')
print("Saved KS p-values to ks_p_values.csv")


### **5.2.3. Plot your balanced dataset**
--------

In [ ]:
# @markdown ##Plot track parameters (balanced dataset)

# Parameters to adapt in function of the notebook section
base_folder = f"{Results_Folder}/Balanced_dataset/track_parameters_plots"
Conditions = 'Condition'
df_to_plot = balanced_merged_tracks_df

# Check and create necessary directories
folders = ["pdf", "csv"]
for folder in folders:
    dir_path = os.path.join(base_folder, folder)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

condition_selector, condition_accordion = display_condition_selection(df_to_plot, Conditions)
checkboxes_dict, checkboxes_accordion = display_variable_checkboxes(categorize_columns(df_to_plot))
variable_checkboxes, checkboxes_widget = display_variable_checkboxes(get_selectable_columns_plots(df_to_plot))
stat_method_selector = widgets.Dropdown(
    options=['randomization test', 't-test'],
    value='randomization test',
    description='Stat Method:',
    style={'description_width': 'initial'}
)

button = Button(description="Plot Selected Variables", layout=Layout(width='400px'), button_style='info')
button.on_click(lambda b: plot_selected_vars(b, checkboxes_dict, df_to_plot, Conditions, base_folder, condition_selector, stat_method_selector));

display(VBox([condition_accordion, checkboxes_accordion, stat_method_selector, button]))

# **Part 6. Version log**
---
While we strive to provide accurate and helpful information, please be aware that:
  - This notebook may contain bugs.
  - Features are currently limited and will be expanded in future releases.

We encourage users to report any issues or suggestions for improvement. Please check the [repository](https://github.com/guijacquemet/CellTracksColab) regularly for updates and the latest version of this notebook.

#### **Known Issues**:
- Tracks are displayed in 2D in section 1.4

**Version 1.1.0**
  - Make the notebook locally usable:
    - Replace Google Colab's parameter widgets with ipywidgets
    - Merge the initial Google Colab related code with the imports code cell
    - Remove unnecesary or repeated imports
    - Include additional functionalities for local vs Colab usage
    - Update notebook versioning checking
    
**Version 1.0.1**
  - Includes a general data reader
  - Plotting functions are imported from the main code

**Version 0.9.1**
  - Added the PIP freeze option to save a requirement text
  - Added the heatmap visualisation of track parameters
  - Heatmaps can now be displayed on multiple pages
  - Fix userwarning message during plotting (all box plots)
  - Added the possibility to copy and paste an existing list of selected metric for clustering analyses

**Version 0.9**
  - Improved plotting strategy. Specific conditions can be chosen
  - absolute cohen d values are now shown
  - In the QC the heatmap is automatically divided in subplot when too many columns are in the df

**Version 0.8**
  - Settings are now saved
  - Order of the section has been modified to help streamline biological discoveries
  - New section added to quality Control to check if the dataset is balanced
  - New section added to the UMAP and tsne section to plot track parameters for selected clusters
  - clusters for UMAP and t-sne are now saved in the dataframe separetly

**Version 0.7**
  - check_for_nans function added
  - Clustering using t-SNE added

**Version 0.6**
  - Improved organisation of the results
  - Tracks visualisation are now saved

**Version 0.5**
  - Improved part 5
  - Added the possibility to find examplar on the raw movies when available
  - Added the possibility to export video with the examplar labeled
  - Code improved to deal with larger dataset (tested with over 50k tracks)
  - test dataset now contains raw video and is hosted on Zenodo
  - Results are now organised in folders
  - Added progress bars
  - Minor code fixes

**Version 0.4**

  - Added the possibility to filter and smooth tracks
  - Added spatial and temporal calibration
  - Notebook is streamlined
  - multiple bug fix
  - Remove the t-sne
  - Improved documentation

**Version 0.3**
  - Fix a nasty bug in the import functions
  - Add basic examplar for UMAP
  - Added the statistical analyses and their explanations.
  - Added a new quality control part that helps assessing the similarity of results between FOV, conditions and repeats
  - Improved part 5 (previously part 4).

**Version 0.2**
  - Added support for 3D tracks
  - New documentation and metrics added.

**Version 0.1**
This is the first release of this notebook.

---